# proventhru: M1 in Colab

M1 (`PROTOCOL.md` v3) is `Qwen/Qwen2.5-7B-Instruct`, pinned by revision hash, in bf16 under vLLM on an **A100**. It runs against coqtop with **Coq 8.18.0**, and the run checks the prover version, so a run on any other Coq is refused. Everything happens here, the baseline reruns included.

Runtime: **A100 GPU** (cell 5 refuses any other: the baseline, DPO and GRPO all have to run on the same GPU class). Run the cells in order. Cells 1–3 are setup. The first session takes about 20–25 minutes, because the opam build is cached to Drive; later sessions take about 3.

What this notebook may do under the protocol now: run model policies on the **dev** set (smoke test, prompt tuning). Held-out runs are refused until the freezing amendment registers the prompt hash and the model revision printed by cell 4, together with the vLLM version and GPU printed by cell 5.

If installing vLLM (cell 5) replaces Colab's PyTorch and asks for a runtime restart, restart and rerun from cell 1. The cached Coq unpacks in about a minute, and the pinned versions are read back from Drive.

In [ ]:
# 1. Drive, and where everything persistent lives
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
ROOT = '/content/drive/MyDrive/proventhru'
for d in ('', 'opam', 'runs', 'cache'):
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)
BRANCH = 'experiment-protocol'   # the branch this notebook belongs to

In [ ]:
%%bash -s "$ROOT"
# 2. Coq 8.18.0 with opam. Built once, then cached on Drive as a tarball:
#    Drive is too slow to hold an opam root directly, so it is unpacked to local disk.
set -e
ROOT="$1"; TAR="$ROOT/opam/opam-coq-8.18.0.tar.zst"
apt-get -qq update >/dev/null && apt-get -qq install -y opam zstd build-essential m4 libgmp-dev pkg-config >/dev/null
if [ -f "$TAR" ]; then
  echo "unpacking cached opam switch"; tar -C / --zstd -xf "$TAR"
else
  echo "building Coq 8.18.0 (about 15-20 minutes, once)"
  opam init -y --bare --no-setup --disable-sandboxing -q
  opam switch create coq818 4.14.1 -y -q
  opam install -y -q --switch coq818 coq.8.18.0
  tar -C / --zstd -cf "$TAR" root/.opam
fi
eval "$(opam env --switch coq818 --set-switch)"
coqtop -v | head -n 1

In [ ]:
# 3. proventhru at this branch, Coq on PATH, and the coqtop tests as a smoke check
import os, subprocess
opam_bin = subprocess.run(['opam', 'var', 'bin', '--switch', 'coq818'], capture_output=True, text=True).stdout.strip()
os.environ['PATH'] = opam_bin + ':' + os.environ['PATH']
if not os.path.exists('/content/proventhru'):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, 'https://github.com/bigDSanalyst/proventhru', '/content/proventhru'], check=True)
%cd /content/proventhru
!git log -1 --format='proventhru commit %H'
!pip -q install -e .
!coqtop -v | head -n 1
!python -m unittest tests.test_experiment 2>&1 | tail -n 3
!python -m proventhru.cli protocol check PROTOCOL.md | head -n 1

In [ ]:
# 4. M1's weights, pinned: resolve the revision once, then always load that revision.
from huggingface_hub import HfApi, snapshot_download
MODEL = 'Qwen/Qwen2.5-7B-Instruct'
REV_FILE = f'{ROOT}/m1_revision.txt'
if os.path.exists(REV_FILE):
    REVISION = open(REV_FILE).read().strip()
else:
    REVISION = HfApi().model_info(MODEL).sha
    open(REV_FILE, 'w').write(REVISION)
WEIGHTS = snapshot_download(MODEL, revision=REVISION)
SERVED = f'{MODEL}@{REVISION}'
print('M1 revision:', REVISION)
print('served as  :', SERVED, '  <- this goes into the freezing amendment')

In [ ]:
# 5. vLLM serving M1 in bf16 on an A100. The vLLM version is resolved once and kept on Drive,
#    like the model revision: every later session, and the DPO/GRPO stages, use the same one.
import subprocess, time, urllib.request, json, os
GPU = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
assert 'A100' in GPU, f'M1 runs on an A100 only (PROTOCOL.md v3); this runtime has: {GPU}'
VLLM_FILE = f'{ROOT}/vllm_version.txt'
if os.path.exists(VLLM_FILE):
    VLLM = open(VLLM_FILE).read().strip()
    !pip -q install vllm=={VLLM}
else:
    !pip -q install vllm
    VLLM = subprocess.run(['python', '-c', 'import vllm; print(vllm.__version__)'], capture_output=True, text=True).stdout.strip()
    open(VLLM_FILE, 'w').write(VLLM)
server = subprocess.Popen(
    ['python', '-m', 'vllm.entrypoints.openai.api_server', '--model', WEIGHTS,
     '--served-model-name', SERVED, '--dtype', 'bfloat16', '--max-model-len', '8192',
     '--seed', '0', '--gpu-memory-utilization', '0.85', '--port', '8000'],
    stdout=open('/content/vllm.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(240):
    try:
        print(json.load(urllib.request.urlopen('http://localhost:8000/v1/models'))['data'][0]['id']); break
    except Exception:
        time.sleep(5)
else:
    print(open('/content/vllm.log').read()[-3000:])
PROVIDER = f'vllm {VLLM}, bf16, {GPU}'
print(PROVIDER, '  <- this goes into the freezing amendment')

In [ ]:
# 6a. Smoke test: 10 calls, one per dev statement, at the root state. No search, no record.
#     Answered? Parses against the schema? Candidates in the vocabulary? Tokens per call?
#     If json_schema fails on this vLLM, rerun with --response-format json_object and use that everywhere.
!python tools/smoke.py --base-url http://localhost:8000/v1 --model "{SERVED}" --key-env '' \
    --response-format json_schema --n 10 --protocol PROTOCOL.md --json "{ROOT}/smoke-m1.json"


In [ ]:
# 6b. One dev statement searched end to end: the schema-constrained answer comes back and is checked.
#    If json_schema is not supported by this vLLM, use --response-format json_object instead
#    (and use that same setting for every run: it is part of the policy's identity).
!python -m proventhru.cli --backend coqtop prove "forall n : nat, n * n >= n" --budget 0 --step-budget 60 \
    --policy openai --base-url http://localhost:8000/v1 --key-env '' --model "{SERVED}" \
    --response-format json_schema --provider "{PROVIDER}" --protocol PROTOCOL.md --trace | tail -n 15

In [ ]:
# 7. Dev set, 600 steps, 16 statements at once (vLLM batches their calls).
#    Prompt tuning happens here and only here. The cache makes reruns free; the run resumes if interrupted.
from proventhru.policy_openai import prompt_sha256
OUT = f'{ROOT}/runs/dev-C600-p{prompt_sha256()[:8]}'   # one record per prompt version
!python -m proventhru.cli --backend coqtop run examples/eval_open.txt --out "{OUT}" --budget 0 --step-budget 600 \
    --jobs 16 --policy openai --base-url http://localhost:8000/v1 --key-env '' --model "{SERVED}" \
    --response-format json_schema --provider "{PROVIDER}" --cache "{ROOT}/cache/m1.jsonl" --protocol PROTOCOL.md
!python -m proventhru.cli report "{OUT}/records.jsonl"

## Held-out runs (after the freezing amendment)

Once the prompt is frozen and `PROTOCOL.md` registers `SERVED` and the prompt hash, run these in order:

1. A and B on `test`, rerun here under the same version. They must reproduce v1's 22 and 22 at 600 steps (and 26 and 32 at 2000).
2. C and D on `test`, and C on `test_renamed`, each at `--step-budget 600`.
3. M1's C and D on `test` at `--step-budget 2000`.

Until then the model commands are refused by the protocol check, which is intended. The baseline cell runs now, but run it after the freeze, so its records cite the same version as the model runs.

In [ ]:
# Baselines, rerun under the version the model runs cite (CPU only; jobs = Colab's cores).
# for name, budget, extra in (('A600', 600, ''), ('B600', 600, '--retrieval 6'), ('A2000', 2000, ''), ('B2000', 2000, '--retrieval 6')):
#     !python -m proventhru.cli --backend coqtop run examples/eval_test.txt --out "{ROOT}/runs/test-{name}" --budget 0 \
#         --step-budget {budget} --jobs {os.cpu_count()} --protocol PROTOCOL.md {extra}

# Model runs.
# for c, extra in (('C600', ''), ('D600', '--retrieval 6')):
#     !python -m proventhru.cli --backend coqtop run examples/eval_test.txt --out "{ROOT}/runs/test-{c}" --budget 0 \
#         --step-budget 600 --jobs 16 --policy openai --base-url http://localhost:8000/v1 --key-env '' --model "{SERVED}" \
#         --response-format json_schema --provider "{PROVIDER}" --cache "{ROOT}/cache/m1.jsonl" --protocol PROTOCOL.md {extra}